In [ ]:
import csv

class JobSeeker:
    def __init__(self, skills, experience, salary, location, job_interest, sector, education_level, job_id):
        self.skills = skills
        self.experience = experience
        self.salary = salary
        self.location = location
        self.job_interest = job_interest
        self.sector = sector
        self.education_level = education_level
        self.job_id = job_id

    def __repr__(self):
        return (f"JobSeeker(job_id={self.job_id}, skills={self.skills}, "
                f"experience={self.experience}, salary={self.salary}, "
                f"location={self.location}, job_interest={self.job_interest}, "
                f"sector={self.sector}, education_level={self.education_level})")

class JobOffer:
    def __init__(self, required_skills, min_experience, salary_range, location, sector, education_level):
        self.required_skills = required_skills
        self.min_experience = min_experience
        self.salary_range = salary_range
        self.location = location
        self.sector = sector
        self.education_level = education_level

    def __repr__(self):
        return (f"JobOffer(required_skills={self.required_skills}, "
                f"min_experience={self.min_experience}, "
                f"salary_range={self.salary_range}, "
                f"location={self.location}, "
                f"sector={self.sector}, "
                f"education_level={self.education_level})")

def load_job_seekers(filename):
    job_seekers = []
    try:
        with open(filename, mode='r') as file:
            reader = csv.DictReader(file)
            for row in reader:
                job_seekers.append(JobSeeker(
                    skills=row['skills'].split(', '),
                    experience=int(row['experience']),
                    salary=int(row['salary']),
                    location=row['location'],
                    job_interest=row['job_interest'],
                    sector=row['sector'],
                    education_level=row['education_level'],
                    job_id=row['job_id']
                ))
        print(f"Loaded {len(job_seekers)} job seekers.")
    except FileNotFoundError:
        print(f"Error: The file {filename} was not found.")
    except Exception as e:
        print(f"Error: {e}")
    return job_seekers

def load_job_offers(filename):
    job_offers = []
    try:
        with open(filename, mode='r') as file:
            reader = csv.DictReader(file)
            for row in reader:
                # Parse salary range (e.g., "50000-70000" becomes (50000, 70000))
                salary_min, salary_max = map(int, row['salary_range'].split('-'))
                job_offers.append(JobOffer(
                    required_skills=row['required_skills'].split(', '),
                    min_experience=int(row['min_experience']),
                    salary_range=(salary_min, salary_max),
                    location=row['location'],
                    sector=row['sector'],
                    education_level=row['education_level']
                ))
        print(f"Loaded {len(job_offers)} job offers.")
    except FileNotFoundError:
        print(f"Error: The file {filename} was not found.")
    except KeyError as e:
        print(f"CSV column missing: {e}")
    except ValueError as e:
        print(f"Data format error: {e}")
    except Exception as e:
        print(f"Error: {e}")
    return job_offers

def improved_heuristic(job_seeker, job_offer):
    missing_skills = set(job_offer.required_skills) - set(job_seeker.skills)
    skill_penalty = len(missing_skills) * 10

    experience_penalty = max(0, job_offer.min_experience - job_seeker.experience) * 5

    salary_penalty = 0
    if job_seeker.salary < job_offer.salary_range[0]:
        salary_penalty = (job_offer.salary_range[0] - job_seeker.salary) / 1000
    elif job_seeker.salary > job_offer.salary_range[1]:
        salary_penalty = (job_seeker.salary - job_offer.salary_range[1]) / 1000

    location_penalty = 0 if job_seeker.location.lower() == job_offer.location.lower() else 15

    education_levels = {
        "no formal education": 0,
        "high school": 1,
        "technical diploma": 2,
        "bachelor's": 3,
        "bachelor": 3,
        "ingénieur": 4,
        "master's": 5,
        "master": 5,
        "doctorate": 6,
        "phd": 6
    }

    seeker_edu_level = education_levels.get(job_seeker.education_level.lower(), 0)
    job_edu_level = education_levels.get(job_offer.education_level.lower(), 0)

    if seeker_edu_level < job_edu_level:
        education_penalty = (job_edu_level - seeker_edu_level) * 12
    else:
        overqualification = max(0, seeker_edu_level - job_edu_level)
        education_penalty = overqualification * 3
        if job_edu_level == 3 and overqualification > 0:
            education_penalty = overqualification * 8

    sector_penalty = 0
    if job_seeker.sector.lower() != job_offer.sector.lower():
        related_sectors = {
            "tech": ["information technology", "software", "it"],
            "information technology": ["tech", "software", "it"],
            "business": ["finance", "marketing", "consulting"],
            "finance": ["business", "banking", "accounting"],
            "marketing": ["business", "advertising", "sales"],
            "healthcare": ["medical", "pharma", "health"],
        }
        if (job_offer.sector.lower() in related_sectors.get(job_seeker.sector.lower(), []) or \
           (job_seeker.sector.lower() in related_sectors.get(job_offer.sector.lower(), [])):
            sector_penalty = 8
        else:
            sector_penalty = 18

    interest_penalty = 0
    if job_seeker.job_interest.lower() != job_offer.sector.lower():
        interest_penalty = 5

    total_penalty = (skill_penalty + experience_penalty + salary_penalty +
                     location_penalty + education_penalty + sector_penalty +
                     interest_penalty)
    return total_penalty

def greedy_match(job_seekers, job_offers):
    matched_seeker_ids = set()
    matches = []
    
    for offer in job_offers:
        best_seeker = None
        min_score = float('inf')
        
        for seeker in job_seekers:
            if seeker.job_id not in matched_seeker_ids:
                current_score = improved_heuristic(seeker, offer)
                if current_score < min_score:
                    min_score = current_score
                    best_seeker = seeker
        
        if best_seeker:
            matches.append((offer, best_seeker, min_score))
            matched_seeker_ids.add(best_seeker.job_id)
    
    return matches

def main():
    # Load job seekers (fallback to sample data if CSV not found)
    job_seekers = load_job_seekers("job_seekers_with_ids.csv")
    if not job_seekers:
        print("\nUsing sample job seekers:")
        job_seekers = [
            JobSeeker(
                skills=['Copywriting', 'SEO'],
                experience=4,
                salary=60000,
                location="Algiers",
                job_interest="Tech",
                sector="Tech",
                education_level="Bachelor",
                job_id='101'
            ),
            JobSeeker(
                skills=['Java', 'Data Analysis'],
                experience=2,
                salary=50000,
                location="Tizi Ouzou",
                job_interest="Tech",
                sector="Tech",
                education_level="Master's",
                job_id='102'
            ),
            JobSeeker(
                skills=['SEO', 'Google Ads'],
                experience=3,
                salary=70000,
                location="Algiers",
                job_interest="Business",
                sector="Business",
                education_level="Master's",
                job_id='103'
            ),
            JobSeeker(
                skills=['Java', 'SQL'],
                experience=2,
                salary=45000,
                location="Tizi Ouzou",
                job_interest="Information Technology",
                sector="Information Technology",
                education_level="Bachelor's",
                job_id='104'
            )
        ]
        for seeker in job_seekers:
            print(f" - {seeker}")

    # Load job offers from CSV (mandatory)
    job_offers = load_job_offers("job_offers.csv")
    if not job_offers:
        print("Error: No job offers loaded. Exiting.")
        return

    # Perform greedy matching
    matches = greedy_match(job_seekers, job_offers)
    
    # Display results
    print("\nMatching Results:")
    for offer, seeker, score in matches:
        print(f"\nJob Offer: {offer}")
        print(f"Matched Job Seeker: {seeker.job_id}")
        print(f"Heuristic Score: {score:.2f}")
        print("=" * 50)

if __name__ == "__main__":
    main()